#### Necessary Imports

In [1]:
import requests, json
from PIL import Image
import io, os

StatementMeta(, 8b222e13-c13a-43cd-9dbb-699e5802898a, 3, Finished, Available, Finished, False)

#### Credentials from Key Vault

In [2]:
'''
vault_url     = "https://my-key-resource.vault.azure.net/"
api_key       = notebookutils.credentials.getSecret(vault_url, "ai-vision-api-key")
endpoint      = notebookutils.credentials.getSecret(vault_url, "ai-vision-endpoint")
'''

StatementMeta(, 8b222e13-c13a-43cd-9dbb-699e5802898a, 4, Finished, Available, Finished, False)

'\nvault_url     = "https://my-key-resource.vault.azure.net/"\napi_key       = notebookutils.credentials.getSecret(vault_url, "ai-vision-api-key")\nendpoint      = notebookutils.credentials.getSecret(vault_url, "ai-vision-endpoint")\n'

In [3]:
vault_url     = "https://cv-training-key.vault.azure.net/"
api_key       = notebookutils.credentials.getSecret(vault_url, "det-obj-key")
endpoint      = notebookutils.credentials.getSecret(vault_url, "det-obj-endpoint")

StatementMeta(, 8b222e13-c13a-43cd-9dbb-699e5802898a, 5, Finished, Available, Finished, False)

#### Image analyzer

In [4]:
# Tagged Parameters cell
image_path = "test-object-detection/images.jpeg"  

StatementMeta(, 8b222e13-c13a-43cd-9dbb-699e5802898a, 6, Finished, Available, Finished, False)

In [5]:
# --- Copy image from Lakehouse to /tmp/ ---
files_listing  = notebookutils.fs.ls("Files")
lakehouse_root = files_listing[0].path.split("/Files/")[0]
abs_image_path = f"{lakehouse_root}/Files/{image_path}"
notebookutils.fs.cp(abs_image_path, "file:/tmp/input.jpg")

StatementMeta(, 8b222e13-c13a-43cd-9dbb-699e5802898a, 7, Finished, Available, Finished, False)

True

#### Call Azure AI Vision Object Detection AP

In [11]:
detect_url = f"{endpoint}/computervision/imageanalysis:analyze"

params  = {
    "features": "objects,tags",   # objects returns bounding boxes
    "api-version": "2024-02-01"
}
headers = {
    "Ocp-Apim-Subscription-Key": api_key,
    "Content-Type": "application/octet-stream"
}

with open("/tmp/input.jpg", "rb") as img:
    response = requests.post(detect_url, params=params, headers=headers, data=img)

result = response.json()

StatementMeta(, 8b222e13-c13a-43cd-9dbb-699e5802898a, 13, Finished, Available, Finished, False)

In [12]:
print(result)

StatementMeta(, 8b222e13-c13a-43cd-9dbb-699e5802898a, 14, Finished, Available, Finished, False)

{'modelVersion': '2023-10-01', 'metadata': {'width': 455, 'height': 179}, 'tagsResult': {'values': [{'name': 'sky', 'confidence': 0.9684250950813293}, {'name': 'smile', 'confidence': 0.9613847136497498}, {'name': 'outdoor', 'confidence': 0.937747597694397}, {'name': 'person', 'confidence': 0.9347834587097168}, {'name': 'dog breed', 'confidence': 0.9145358800888062}, {'name': 'grass', 'confidence': 0.8281559348106384}, {'name': 'dog', 'confidence': 0.8237289190292358}, {'name': 'woman', 'confidence': 0.623397707939148}, {'name': 'girl', 'confidence': 0.5893233418464661}]}, 'objectsResult': {'values': [{'boundingBox': {'x': 157, 'y': 48, 'w': 86, 'h': 127}, 'tags': [{'name': 'retriever', 'confidence': 0.694}]}, {'boundingBox': {'x': 365, 'y': 38, 'w': 88, 'h': 131}, 'tags': [{'name': 'retriever', 'confidence': 0.654}]}, {'boundingBox': {'x': 16, 'y': 50, 'w': 167, 'h': 128}, 'tags': [{'name': 'person', 'confidence': 0.707}]}]}}


#### Filter for animal/pet detections

In [13]:
PET_TAGS  = {"dog", "cat", "animal", "pet", "kitten", "puppy", "canine", "feline"}
THRESHOLD = 0.5

# Step 1 — Check image-level tags for pet presence
image_tags    = {t["name"].lower() for t in result.get("tagsResult", {}).get("values", [])
                 if t["confidence"] >= THRESHOLD}
image_has_pet = bool(image_tags & PET_TAGS)

print(f"Image-level tags found:     {image_tags}")
print(f"Pet detected at image level: {image_has_pet}")

# Step 2 — Filter objects using ALL tags per object including parent categories
detections = []
for obj in result.get("objectsResult", {}).get("values", []):
    obj_tags       = {t["name"].lower() for t in obj.get("tags", [])}
    obj_confidence = obj.get("tags", [{}])[0].get("confidence", 0)
    is_pet         = bool(obj_tags & PET_TAGS)

    if obj_confidence >= THRESHOLD and (is_pet or image_has_pet):
        detections.append(obj)

print(f"Objects matching pet criteria: {len(detections)}")

# Step 3 — Crop or log no detection
if not detections:
    print("⚠️ No pet detected")
else:
    best = max(detections, key=lambda o: o["tags"][0]["confidence"])
    box  = best["boundingBox"]

    with Image.open("/tmp/input.jpg") as img:
        cropped = img.crop((
            box["x"],
            box["y"],
            box["x"] + box["w"],
            box["y"] + box["h"]
        ))
        cropped.save("/tmp/cropped.jpg")

    # Save cropped image permanently to Lakehouse
    filename     = os.path.basename(image_path)
    cropped_dest = f"{lakehouse_root}/Files/development/cropped/{filename}"
    notebookutils.fs.cp("file:/tmp/cropped.jpg", cropped_dest)
    print(f"✅ Pet detected and cropped → saved to: {cropped_dest}")

StatementMeta(, 8b222e13-c13a-43cd-9dbb-699e5802898a, 15, Finished, Available, Finished, False)

Image-level tags found:     {'sky', 'dog breed', 'grass', 'outdoor', 'person', 'woman', 'smile', 'dog', 'girl'}
Pet detected at image level: True
Objects matching pet criteria: 3
✅ Pet detected and cropped → saved to: abfss://a96eab2a-8002-45f2-92b4-d2bf05c2540b@onelake.dfs.fabric.microsoft.com/db748fcd-0cc5-478c-a54c-f4938d542b0f/Files/development/cropped/images.jpeg


#### Save to object_detection_metrics table

In [14]:
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType, TimestampType

filename     = os.path.basename(image_path)
detected     = len(detections) > 0
animal_count = len(detections)
timestamp    = datetime.now()

# Build one row per detected object
if detections:
    rows = []
    for det in detections:
        for tag in det.get("tags", []):
            rows.append(Row(
                image_name    = filename,
                detected      = True,
                animal_count  = animal_count,
                object_name   = tag["name"],
                confidence    = float(tag["confidence"]),
                bbox_x        = int(det["boundingBox"]["x"]),
                bbox_y        = int(det["boundingBox"]["y"]),
                bbox_w        = int(det["boundingBox"]["w"]),
                bbox_h        = int(det["boundingBox"]["h"]),
                timestamp     = timestamp
            ))
else:
    # No animal detected — still log the transaction
    rows = [Row(
        image_name    = filename,
        detected      = False,
        animal_count  = 0,
        object_name   = "none",
        confidence    = 0.0,
        bbox_x        = 0,
        bbox_y        = 0,
        bbox_w        = 0,
        bbox_h        = 0,
        timestamp     = timestamp
    )]

detection_df = spark.createDataFrame(rows)
detection_df.write.mode("append").saveAsTable("object_detection_metrics")

print(f"✅ Detection metrics saved: {animal_count} animal(s) found in '{filename}'")

StatementMeta(, 8b222e13-c13a-43cd-9dbb-699e5802898a, 16, Finished, Available, Finished, False)

✅ Detection metrics saved: 3 animal(s) found in 'images.jpeg'
